# 22 · Query & federation — Tier-2 stores via native clients

**Six Tier-2 stores, six wire protocols, six native clients — and *not one* of them is a
Trino catalog.** Where notebook `20` reached the lakehouse and the operational Postgres
through *one* federated SQL engine, this notebook does the opposite: it speaks to each
store in **its own protocol, with its own driver**, because that is the only way to reach
these six. They are the mesh's specialist stores, each chosen for a shape of data that a
general engine serves poorly.

That distinction is the whole point of this notebook:

> **Notebook `20` federates the two catalog-connected sources through Trino.
> These six are reached *natively* — a different tool for each job.**

### The six stores, and why each exists

| store | data model | native client | reach it for |
|-------|-----------|---------------|--------------|
| **ClickHouse** | columnar OLAP | `clickhouse-connect` (HTTP) | huge scans / aggregations over wide tables |
| **Cassandra** | wide-column | `cassandra-driver` (CQL) | write-heavy, partitioned, linear-scale key access |
| **MongoDB** | document | `pymongo` | schema-flexible nested JSON documents |
| **CockroachDB** | distributed SQL | `psycopg` (pg-wire) | horizontally-scaled, strongly-consistent SQL |
| **TimescaleDB** | time-series | `psycopg` (pg-wire) | time-bucketed metrics over hypertables |
| **MySQL** | relational OLTP | `PyMySQL` | familiar row-oriented transactional tables |

Two of these speak the **Postgres wire protocol** (CockroachDB, TimescaleDB) and one speaks
**MySQL's** — but none is wired into this cluster's Trino, so `SHOW CATALOGS` there never
lists them (notebook `20` says exactly this). To read them you connect *directly*, and that
is what every section below does.

> **Read-only, throughout.** Every query here is a `SELECT` / `find` / `SHOW` / `DESCRIBE`.
> Nothing is inserted, updated, dropped or created in any store, so — as in notebook `20` —
> there is **no cleanup section**. We also never *assume* a table, keyspace, collection or
> database name: each section **discovers what is actually hydrated first**, then queries
> only what it found.

## Setup — install the six native clients

None of these drivers ships in the singleuser base image (which carries `polars`, `s3fs`,
`pyarrow`, `duckdb`, `fastavro`), so we install all six here. `polars` — used to render every
result frame, exactly as in notebooks `20` / `21` — is already in the image. The two pg-wire
stores share one driver (`psycopg`), so it is installed once.

In [1]:
%pip install -q clickhouse-connect cassandra-driver pymongo "psycopg[binary]" PyMySQL


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connection config — env-driven, in-cluster defaults

Every connection is **env-driven**, the same pattern notebooks `20` / `21` use. The committed
defaults are the **in-cluster** service DNS names (`<svc>.data-mesh.svc.cluster.local`); a
validation run overrides each store's `*_HOST` / `*_PORT` via env (e.g. to the store's NodePort)
**without editing the notebook**. So the notebook never captures the resolved address — a
connection is proven by the **server version** it reports back, not by echoing the endpoint.

All six stores are **unmeshed, ClusterIP** services. Four of them —
ClickHouse, MongoDB, TimescaleDB, MySQL — share **one** password, read once from
`TIER2_DB_PASSWORD` (never committed). The other two need no password: **Cassandra** runs
`AllowAllAuthenticator`, and **CockroachDB** is in insecure dev mode (`root`, no password,
`sslmode=disable`). We resolve everything here so each store section stays about the *store*,
not the plumbing.

In [2]:
import os
import polars as pl

# committed defaults = in-cluster DNS; a validation run overrides *_HOST/*_PORT via env
CH_HOST   = os.environ.get("CLICKHOUSE_HOST", "clickhouse.data-mesh.svc.cluster.local")
CH_PORT   = int(os.environ.get("CLICKHOUSE_PORT", "8123"))
CASS_HOST = os.environ.get("CASSANDRA_HOST", "cassandra.data-mesh.svc.cluster.local")
CASS_PORT = int(os.environ.get("CASSANDRA_PORT", "9042"))
MONGO_HOST = os.environ.get("MONGO_HOST", "mongodb.data-mesh.svc.cluster.local")
MONGO_PORT = int(os.environ.get("MONGO_PORT", "27017"))
CRDB_HOST = os.environ.get("COCKROACH_HOST", "cockroachdb.data-mesh.svc.cluster.local")
CRDB_PORT = int(os.environ.get("COCKROACH_PORT", "26257"))
TS_HOST   = os.environ.get("TIMESCALE_HOST", "timescaledb.data-mesh.svc.cluster.local")
TS_PORT   = int(os.environ.get("TIMESCALE_PORT", "5432"))
MY_HOST   = os.environ.get("MYSQL_HOST", "mysql.data-mesh.svc.cluster.local")
MY_PORT   = int(os.environ.get("MYSQL_PORT", "3306"))

# ONE shared password for ClickHouse / Mongo / Timescale / MySQL (Cassandra + Cockroach need none)
PW = os.environ["TIER2_DB_PASSWORD"]

# small helper: rows + column names -> a polars DataFrame for display (house style, as in 20/21)
def frame(rows, cols):
    return pl.DataFrame(rows, schema=list(cols), orient="row")

print("config resolved — six native endpoints, one shared password loaded from $TIER2_DB_PASSWORD")

config resolved — six native endpoints, one shared password loaded from $TIER2_DB_PASSWORD


## 1 · ClickHouse — columnar OLAP

**Reach for ClickHouse when the question is a big aggregation over a wide table.** It stores
columns, not rows, so a `GROUP BY … sum(…)` touches only the columns it names and screams
through millions of rows. It is the mesh's analytical scan engine — the opposite of a
point-lookup store.

We connect over its **HTTP interface** with `clickhouse-connect`, then **discover** the real
databases and tables (skipping the engine's own system databases) before running one columnar
aggregation on a table we confirmed exists.

In [3]:
import clickhouse_connect

ch = clickhouse_connect.get_client(host=CH_HOST, port=CH_PORT, username="default", password=PW)
print(f"connected (ClickHouse {ch.server_version}) — endpoint from env")   # proves the connection, not the address

# discover: user databases (skip ClickHouse's own system schemas), then tables in the music DB
SYS_DBS = {"system", "information_schema", "INFORMATION_SCHEMA", "default"}
databases = [r[0] for r in ch.query("SHOW DATABASES").result_rows if r[0] not in SYS_DBS]
print("\nuser databases      :", databases)

ch_db = next(d for d in databases if d == "datasets_music")
ch_tables = [r[0] for r in ch.query(f"SHOW TABLES FROM {ch_db}").result_rows]
print(f"tables in {ch_db} :", len(ch_tables), "->", ch_tables[:6], "...")

ch_table = next(t for t in ch_tables if t == "fma_tracks")   # resolve, never assume
print("query target        :", f"{ch_db}.{ch_table}")

connected (ClickHouse 24.8.14.39) — endpoint from env

user databases      : ['datasets_health', 'datasets_music', 'langfuse']
tables in datasets_music : 21 -> ['audioset_test', 'audioset_train', 'fma_tracks', 'lp_musiccaps_mc_test', 'lp_musiccaps_mc_train', 'lp_musiccaps_mtt_test'] ...
query target        : datasets_music.fma_tracks


In [4]:
# columnar aggregation: total plays per artist across the Free Music Archive tracks table.
# ClickHouse decodes only the three columns this query names, out of ~50 in the table.
res = ch.query(f"""
    SELECT artist_name,
           count()             AS n_tracks,
           sum(track_listens)  AS total_listens
    FROM {ch_db}.{ch_table}
    WHERE artist_name != ''
    GROUP BY artist_name
    ORDER BY total_listens DESC
    LIMIT 10
""")
frame(res.result_rows, res.column_names)

artist_name,n_tracks,total_listens
str,i64,i64
"""Podington Bear""",604,7287253
"""Chris Zabriskie""",115,6725015
"""Kevin MacLeod""",162,4800471
"""Jahzzar""",330,4545581
"""Kai Engel""",141,3489703
"""Lee Rosevere""",371,2779844
"""Broke For Free""",68,2584777
"""David Szesztay""",158,2176568
"""Josh Woodward""",285,1709388


## 2 · Cassandra — wide-column, partition-first

**Reach for Cassandra when writes are relentless and every read is by partition key.** It is a
wide-column store: data is sharded by a **partition key** and access is fast and linear *within*
a partition, but there are **no server-side joins** and cross-partition aggregation means a full
scan. You design the table around the query, then read one partition at a time.

Two traps make discovery mandatory here:

- **Table names are long and underscored** (the hydration names them after the source dataset),
  so we resolve them from `system_schema` rather than guessing.
- **Empty partition keys were mapped to the literal `__UNKNOWN__`** during hydration. We verify
  the partition key we pick is a real value, not that sentinel, before querying.

We list keyspaces (minus the `system*` ones), list tables in a real keyspace, pick one real
partition, read it with a bounded `fetch_size`, and sort **client-side** (there is no
server-side `ORDER BY` on a non-clustering column — the wide-column model showing through).

In [5]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement

cluster = Cluster([CASS_HOST], port=CASS_PORT)   # AllowAllAuthenticator -> no credentials
cass = cluster.connect()
rel = cass.execute("SELECT release_version FROM system.local").one().release_version
print(f"connected (Cassandra {rel}) — endpoint from env")   # proves the connection, not the address

# discover: non-system keyspaces, then tables in the music keyspace (via system_schema)
keyspaces = [r.keyspace_name for r in cass.execute("SELECT keyspace_name FROM system_schema.keyspaces")
             if not r.keyspace_name.startswith("system")]
print("\nkeyspaces           :", keyspaces)

cass_ks = next(k for k in keyspaces if k == "datasets_music")
cass_tables = [r.table_name for r in
               cass.execute("SELECT table_name FROM system_schema.tables WHERE keyspace_name=%s", (cass_ks,))]
print(f"tables in {cass_ks} :", cass_tables)
cass_table = next(t for t in cass_tables if t == "lastfm")   # long underscored names -> resolve
print("query target        :", f"{cass_ks}.{cass_table}")

# pick ONE real partition key, guarding against the __UNKNOWN__ sentinel from hydration
scan = cass.execute(SimpleStatement(f"SELECT user_id FROM {cass_ks}.{cass_table} LIMIT 50", fetch_size=50))
user_id = next(r.user_id for r in scan if r.user_id != "__UNKNOWN__")
print("chosen partition    : user_id =", user_id[:16], "...")

connected (Cassandra 5.0.8) — endpoint from env

keyspaces           : ['datasets_health', 'datasets_music']
tables in datasets_music : ['lastfm', 'uci_year_prediction']
query target        : datasets_music.lastfm
chosen partition    : user_id = 5e1e43d579802b46 ...


In [6]:
# read ONE partition (all of this listener's scrobbles), then rank client-side.
# fetch_size bounds the page; there is no server-side ORDER BY on play_count (not a clustering col).
stmt = SimpleStatement(
    f"SELECT artist_name, play_count, country FROM {cass_ks}.{cass_table} WHERE user_id = %s",
    fetch_size=200,
)
rows = [{"artist_name": r.artist_name, "play_count": r.play_count, "country": r.country}
        for r in cass.execute(stmt, (user_id,))]
top = pl.DataFrame(rows).sort("play_count", descending=True).head(10)
print(f"{len(rows)} artists in this one partition; top 10 by play_count:")
top

51 artists in this one partition; top 10 by play_count:


artist_name,play_count,country
str,i64,str
"""faithless""",261,"""United Kingdom"""
"""johnny cash""",245,"""United Kingdom"""
"""audioslave""",188,"""United Kingdom"""
"""radiohead""",184,"""United Kingdom"""
"""the smashing pumpkins""",158,"""United Kingdom"""
"""kings of leon""",157,"""United Kingdom"""
"""incubus""",150,"""United Kingdom"""
"""cypress hill""",137,"""United Kingdom"""
"""creed""",133,"""United Kingdom"""


## 3 · MongoDB — document store

**Reach for MongoDB when the record is a nested, schema-flexible document.** No fixed columns,
no migrations to add a field — each row is a JSON-shaped document, and the query language is an
**aggregation pipeline** rather than SQL. It suits semi-structured data whose shape varies
row-to-row.

We connect with `pymongo`, **discover** the non-system databases and the collections inside one,
then run an aggregation pipeline (`$match` → `$group` → `$sort`) on a collection we confirmed is
there. The connection string is built from the resolved parts, so the password never appears as a
literal.

In [7]:
from pymongo import MongoClient

MONGO_AUTHSRC = os.environ.get("MONGO_AUTHSOURCE", "admin")
mongo_uri = f"mongodb://weyland:{PW}@{MONGO_HOST}:{MONGO_PORT}/?authSource={MONGO_AUTHSRC}"
mongo = MongoClient(mongo_uri)
print(f"connected (MongoDB {mongo.server_info()['version']}) — endpoint from env")   # proves the connection, not the address

# discover: non-system databases, then collections in the health database
SYS_MDBS = {"admin", "local", "config"}
mongo_dbs = [d for d in mongo.list_database_names() if d not in SYS_MDBS]
print("\ndatabases           :", mongo_dbs)

mdb = mongo["datasets_health"]
collections = mdb.list_collection_names()
print("collections         :", collections)
mongo_coll = next(c for c in collections if c == "who_gho_adult_obesity")   # resolve, never assume
print("query target        :", f"datasets_health.{mongo_coll}", "-",
      mdb[mongo_coll].estimated_document_count(), "docs")

connected (MongoDB 8.2.11) — endpoint from env

databases           : ['aidlc_kb', 'datasets_health']
collections         : ['open_food_facts', 'who_gho_hypertension', 'who_gho_alcohol_consumption', 'who_gho_tobacco_smoking', 'who_gho_life_expectancy', 'who_gho_healthy_life_expectancy', 'who_gho_diabetes_prevalence', 'who_gho_adult_obesity', 'who_gho_mental_health_disorders']
query target        : datasets_health.who_gho_adult_obesity - 28350 docs


In [8]:
# aggregation pipeline over documents: mean adult-obesity prevalence per WHO region,
# both sexes (Dim1 = SEX_BTSX), ignoring documents with no region or no numeric value.
pipeline = [
    {"$match": {"Dim1": "SEX_BTSX", "NumericValue": {"$ne": None}, "ParentLocation": {"$ne": None}}},
    {"$group": {"_id": "$ParentLocation", "n": {"$sum": 1}, "avg_obesity": {"$avg": "$NumericValue"}}},
    {"$sort": {"avg_obesity": -1}},
]
docs = [{"region": d["_id"], "n": d["n"], "avg_obesity": round(d["avg_obesity"], 2)}
        for d in mdb[mongo_coll].aggregate(pipeline)]
frame([(d["region"], d["n"], d["avg_obesity"]) for d in docs], ["region", "n", "avg_obesity"])

region,n,avg_obesity
str,i64,f64
"""Western Pacific""",1395,27.13
"""Eastern Mediterranean""",990,19.71
"""Americas""",1665,19.44
"""Europe""",2340,16.57
"""Africa""",2115,6.96
"""South-East Asia""",450,4.06


## 4 · CockroachDB — distributed SQL over the Postgres wire

**Reach for CockroachDB when you want ordinary SQL but horizontally scaled and
strongly-consistent** — a database that survives node loss and scales out while still
answering a plain `SELECT`. It speaks the **Postgres wire protocol**, so `psycopg` connects to
it unchanged — but it is **CockroachDB, not Postgres**: the storage underneath is a
distributed, replicated key-value layer.

One CockroachDB detail shows up immediately: like Postgres it **folds unquoted identifiers to
lowercase**, and this data was ingested with mixed-case column names (`"Topic"`, `"Data_value"`),
so those columns must be **double-quoted** to match. We connect (insecure dev mode: `root`, no
password), `SHOW DATABASES` / `SHOW TABLES` to discover, then aggregate a real table.

In [9]:
import psycopg

CRDB_DB = os.environ.get("COCKROACH_DB", "brfss")
crdb = psycopg.connect(host=CRDB_HOST, port=CRDB_PORT, user="root", dbname=CRDB_DB, sslmode="disable")
cur = crdb.cursor()
cur.execute("SELECT version()")
print(f"connected ({cur.fetchone()[0].split(',')[0]}) — endpoint from env")   # proves it is CockroachDB, not Postgres

# discover: databases on the cluster, then tables in the connected database
cur.execute("SHOW DATABASES")
print("\ndatabases           :", [r[0] for r in cur.fetchall()])
cur.execute("SHOW TABLES")
crdb_tables = [r[1] for r in cur.fetchall()]   # SHOW TABLES -> (schema, table, ...)
print(f"tables in {CRDB_DB}     :", crdb_tables)
crdb_table = next(t for t in crdb_tables if t == "brfss_2020")
print("query target        :", f"{CRDB_DB}.{crdb_table}")

connected (CockroachDB CCL v24.2.4 (x86_64-pc-linux-gnu) — endpoint from env



databases           : ['brfss', 'defaultdb', 'nhis', 'postgres', 'system']
tables in brfss     : ['brfss_2020', 'brfss_prevalence_2011_present', 'brfss_prevalence_data', 'brfss_selected_metro', 'brfss_selected_metropolitan_area', 'brfss_smart_metro_2011_present']
query target        : brfss.brfss_2020


In [10]:
# distributed SQL aggregation. Mixed-case ingested columns are double-quoted (Cockroach, like
# Postgres, lowercases bare identifiers). The planner fans this GROUP BY across the cluster's ranges.
cur.execute(f'''
    SELECT "Topic",
           count(*)                        AS n_rows,
           round(avg("Data_value")::numeric, 1) AS avg_value
    FROM {crdb_table}
    WHERE "Data_value" IS NOT NULL
    GROUP BY "Topic"
    ORDER BY n_rows DESC
    LIMIT 10
''')
frame(cur.fetchall(), [d.name for d in cur.description])

Topic,n_rows,avg_value
str,i64,"decimal[38,1]"
"""Disability status""",17504,50.7
"""Cardiovascular Disease""",12878,51.0
"""Arthritis""",9347,38.5
"""Overall Health""",9107,20.2
"""Asthma""",7356,50.1
"""Smoker Status""",7194,25.5
"""BMI Categories""",6312,28.9
"""Last Checkup""",5980,24.1
"""Diabetes""",5103,35.6


## 5 · TimescaleDB — time-series over hypertables

**Reach for TimescaleDB when the data is a stream of timestamped measurements.** It is
Postgres with a **hypertable** extension: a table transparently partitioned by time into
chunks, plus time-native SQL like `time_bucket()` for downsampling. Same pg-wire, same
`psycopg` — but the query shape is temporal.

We connect to the `timeseries` database, **discover an actual hypertable** from
`timescaledb_information.hypertables` (rather than assuming one), and run a `time_bucket`
rollup over it. The mesh feeds several operational metric streams in here; we bucket the
RAG-eval judge scores by day.

In [11]:
TS_DB = os.environ.get("TIMESCALE_DB", "timeseries")
ts = psycopg.connect(host=TS_HOST, port=TS_PORT, user="weyland", password=PW, dbname=TS_DB, sslmode="disable")
tcur = ts.cursor()
tcur.execute("SELECT extversion FROM pg_extension WHERE extname = 'timescaledb'")
print(f"connected (TimescaleDB ext {tcur.fetchone()[0]}) — endpoint from env")   # proves the hypertable extension is live

# discover: real hypertables (not plain tables), then pick one to bucket
tcur.execute("SELECT hypertable_name FROM timescaledb_information.hypertables ORDER BY 1")
hypertables = [r[0] for r in tcur.fetchall()]
print("\nhypertables         :", hypertables)
ts_table = next(h for h in hypertables if h == "eval_scores_ts")   # resolve, never assume
print("query target        :", f"{TS_DB}.{ts_table} (hypertable)")

connected (TimescaleDB ext 2.28.3) — endpoint from env

hypertables         : ['dagster_run_durations', 'datahub_ingestion_runs', 'eval_scores_ts', 'guardrail_verdicts_ts', 'unleash_feature_metrics', 'who_gho_adult_obesity', 'who_gho_alcohol_consumption', 'who_gho_diabetes_prevalence', 'who_gho_healthy_life_expectancy', 'who_gho_hypertension', 'who_gho_life_expectancy', 'who_gho_mental_health_disorders', 'who_gho_tobacco_smoking']
query target        : timeseries.eval_scores_ts (hypertable)


In [12]:
# time-series rollup: bucket the judge scores into 1-day windows per metric.
# time_bucket() is TimescaleDB's native downsampler; a plain Postgres table has no such function.
tcur.execute(f"""
    SELECT time_bucket('1 day', time)       AS day,
           metric,
           count(*)                         AS n,
           round(avg(score)::numeric, 3)    AS avg_score
    FROM {ts_table}
    GROUP BY day, metric
    ORDER BY day DESC, metric
    LIMIT 15
""")
frame(tcur.fetchall(), [d.name for d in tcur.description])

day,metric,n,avg_score
"datetime[μs, Etc/UTC]",str,i64,"decimal[38,3]"
2026-08-22 00:00:00 UTC,"""answer_relevancy""",3570,0.878
2026-08-22 00:00:00 UTC,"""context_relevancy""",3560,0.836
2026-08-22 00:00:00 UTC,"""faithfulness""",3570,0.837
2026-08-08 00:00:00 UTC,"""answer_relevancy""",8640,0.866
2026-08-08 00:00:00 UTC,"""context_relevancy""",8640,0.826
…,…,…,…
2026-07-27 00:00:00 UTC,"""context_relevancy""",18000,0.541
2026-07-27 00:00:00 UTC,"""faithfulness""",18000,0.654
2026-07-25 00:00:00 UTC,"""answer_relevancy""",9000,0.736


## 6 · MySQL — relational OLTP

**Reach for MySQL when you want the familiar, battle-tested row-oriented relational engine** —
ordinary tables, ordinary SQL, the default many services already speak. We connect with
`PyMySQL`.

Discovery matters here for an honest reason: **there is no single `health` database.** The
health datasets are spread across several MySQL schemas (`who_gho`, `brfss`, `nhanes`, `nhis`,
`big_five`, `cdc_physical_activity`) — so we `SHOW DATABASES` first, then work inside one real
schema (`who_gho`) rather than assuming a catch-all name that does not exist.

In [13]:
import pymysql

MY_DB = os.environ.get("MYSQL_DB", "who_gho")
my = pymysql.connect(host=MY_HOST, port=MY_PORT, user="weyland", password=PW, database=MY_DB)
mycur = my.cursor()
mycur.execute("SELECT version()")
print(f"connected (MySQL {mycur.fetchone()[0]}) — endpoint from env")   # proves the connection, not the address

# discover: user databases (note the health domain is spread across several, not one 'health' db)
mycur.execute("SHOW DATABASES")
SYS_MYDBS = {"information_schema", "performance_schema", "mysql", "sys"}
my_dbs = [r[0] for r in mycur.fetchall() if r[0] not in SYS_MYDBS]
print("\nuser databases      :", my_dbs)
mycur.execute("SHOW TABLES")
my_tables = [r[0] for r in mycur.fetchall()]
print(f"tables in {MY_DB}    :", my_tables)
my_table = next(t for t in my_tables if t == "adult_obesity")   # resolve, never assume
print("query target        :", f"{MY_DB}.{my_table}")

connected (MySQL 8.4.10) — endpoint from env

user databases      : ['big_five', 'brfss', 'cdc_physical_activity', 'nhanes', 'nhis', 'who_gho']
tables in who_gho    : ['adult_obesity', 'alcohol_consumption', 'diabetes_prevalence', 'healthy_life_expectancy', 'hypertension', 'life_expectancy', 'mental_health_disorders', 'tobacco_smoking']
query target        : who_gho.adult_obesity


In [14]:
# relational aggregation: mean adult-obesity prevalence per WHO region, both sexes.
# Ordinary SQL on an ordinary row-store -- the same shape a service's OLTP query would take.
mycur.execute(f"""
    SELECT ParentLocation,
           count(*)                    AS n_rows,
           round(avg(NumericValue), 2) AS avg_obesity
    FROM {my_table}
    WHERE Dim1 = 'SEX_BTSX' AND NumericValue IS NOT NULL
    GROUP BY ParentLocation
    ORDER BY avg_obesity DESC
""")
frame(mycur.fetchall(), [d[0] for d in mycur.description])

ParentLocation,n_rows,avg_obesity
str,i64,f64
"""Western Pacific""",1395,27.13
"""Eastern Mediterranean""",990,19.71
"""Americas""",1665,19.44
"""Europe""",2340,16.57
null,495,10.4
"""Africa""",2115,6.96
"""South-East Asia""",450,4.06


## When to reach for each — and where notebook `20` fits

All six stores live in the same mesh, but each earns its place by a **shape of data** the
others serve poorly. That is why none is a Trino catalog and each needs its own client: the
right tool is different per job.

| store | data model | reach for it when… | the one situation it wins |
|-------|-----------|--------------------|---------------------------|
| **ClickHouse** | columnar OLAP | you scan/aggregate wide tables | a `GROUP BY sum(...)` over millions of rows, touching a few columns |
| **Cassandra** | wide-column | writes are relentless, reads are by key | linear-scale partitioned access, no joins needed |
| **MongoDB** | document | records are nested and schema-varies | semi-structured JSON documents with no fixed columns |
| **CockroachDB** | distributed SQL | you want SQL that survives node loss | strongly-consistent SQL scaled horizontally |
| **TimescaleDB** | time-series | data is a stream of measurements | `time_bucket` rollups over time-partitioned hypertables |
| **MySQL** | relational OLTP | you want the familiar row store | ordinary transactional tables and plain SQL |

**And the framing against the rest of the query wave.** The query layer reaches the mesh's
stores three ways, and this notebook is the third:

- **Notebook `20` — Trino** federates the **two catalog-connected sources** (the Iceberg
  lakehouse and the operational Postgres) — *one* SQL statement across both, no ETL. It reaches
  exactly those two; `SHOW CATALOGS` there lists only `iceberg`, `postgresql`, `system`.
- **Notebook `21` — DuckDB / GizmoSQL** is the single-node OLAP engine — embedded in your process
  over lakeFS Parquet, or served to many clients over Arrow Flight SQL.
- **Notebook `22` — this one** reaches the **six specialist Tier-2 stores natively**, because
  they are *not* federated by that Trino and each speaks its own protocol.

> **Federate the catalog-connected pair → Trino (`20`). Drive one columnar engine hard →
> DuckDB (`21`). Reach a specialist store on its own protocol → its native client (this
> notebook, `22`).** Together, `20` + `21` + `22` are the complete query/federation wave — the
> whole set of ways the mesh answers a question, from one federated `JOIN` down to six
> purpose-built native drivers.